In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms

In [4]:
from PIL import Image
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

In [5]:
#check device
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)

using device: cpu


In [6]:
#image transformation
transform=transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

In [7]:
# load cats and dogs dataset
dataset_path="./kagglecatsanddogs_5340/PetImages"

dataset=ImageFolder(
    root=dataset_path,
    transform=transform
)

print("Classes:", dataset.classes)
print("Class mapping:", dataset.class_to_idx)
print("Total Images:", len(dataset))

Classes: ['Cat', 'Dog']
Class mapping: {'Cat': 0, 'Dog': 1}
Total Images: 24998


In [8]:
print(os.path.exists("./kagglecatsanddogs_5340/PetImages"))

True


In [9]:
# split dataset into training and validation data

train_size=int(0.8*len(dataset))
validation_size=len(dataset)-train_size

train_data, validation_data = random_split(
    dataset,
    [train_size, validation_size]
)

print("Training images:", len(train_data))
print("Validation images:", len(validation_data))

Training images: 19998
Validation images: 5000


In [10]:
# create dataloaders

batch_size=32

trainloader=DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True
)

validationloader=DataLoader(
    validation_data,
    batch_size=batch_size,
    shuffle=False
)

In [11]:
#build the CNN

class CatsDogsCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_layers=nn.Sequential(
            #input=3*128*128
            nn.Conv2d(
                in_channels=3,
                out_channels=16,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            #output=16*64*64
            nn.Conv2d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            #output=32*32*32
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            #output=64*16*16
        )
        self.fc_layers=nn.Sequential(
            nn.Flatten(),

            # 64*16*16
            nn.Linear(64*16*16, 128), #128=no of neurons
            nn.ReLU(),

            nn.Dropout(0.5),

            # one output for binary classification
            nn.Linear(128, 1)
        )
    def forward(self, x):
        x=self.conv_layers(x)
        x=self.fc_layers(x)
        return x

In [12]:
# create model
model=CatsDogsCNN().to(device)
print(model)

CatsDogsCNN(
  (conv_layers): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layers): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=16384, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=128, out_features=1, bias=True)
  )
)


In [13]:
# define the loss function and optimizer

criterion=nn.BCEWithLogitsLoss()

optimizer=optim.Adam(
    model.parameters(),
    lr=0.001
)

In [25]:
# training function

def train_model(model, trainloader, validationloader, epochs):
    best_validation_loss=float("inf")
    for epoch in range(epochs):
        model.train()

        total_train_loss=0
        correct_train=0
        total_train=0

        for images, labels in trainloader:
            images=images.to(device)

            labels=labels.float().unsqueeze(1).to(device) # convert labels into float and shae[batch_size,1]

            optimizer.zero_grad() # clear old gradients

            outputs=model(images) # forward pass

            loss=criterion(outputs, labels) # calculate loss

            loss.backward() # backpropagation

            optimizer.step() # update weigths and biases

            total_train_loss += loss.item()

            probabilities=torch.sigmoid(outputs) # convert logits into probabilities

            predictions=(probabilities>=0.5).float() # probab >= 0.5 means dog also it is converted to float becoz probab >= 0.5 returns true or false and converting to float means f=0.0 and t=1.0

            correct_train += (predictions == labels).sum().item()

            total_train += labels.size(0)

        average_train_loss=total_train_loss/len(trainloader)
        train_accuracy=100*correct_train/total_train

        # validation

        model.eval()

        total_validation_loss=0
        correct_validation=0
        total_validation=0

        with torch.no_grad():
            for images, labels in validationloader:
                images=images.to(device)
                labels=labels.float().unsqueeze(1).to(device)

                outputs=model(images)

                loss=criterion(outputs, labels)

                total_validation_loss+=loss.item()

                probabilities=torch.sigmoid(outputs)

                predictions=(probabilities >= 0.5).float()

                correct_validation+=(predictions==labels).sum().item()

                total_validation+=labels.size(0)

        average_validation_loss = (
            total_validation_loss / len(validationloader)
        )

        validation_accuracy = (
            100*correct_validation / total_validation
        )

        # print results
        print(
            f"Epoch [{epoch + 1}/{epochs}] | "
            f"Train Loss: {average_train_loss:.4f} | "
            f"Train Accuracy: {train_accuracy:.2f}% | "
            f"Validation Loss: {average_validation_loss:.4f} | "
            f"Validation Accuracy: {validation_accuracy:.2f}%"
        )

        #saving the best model
        if average_validation_loss < best_validation_loss:
            best_validation_loss=average_validation_loss
            torch.save(model.state_dict(), "best_model.pt")
        

In [26]:
from PIL import Image
import os

folders = [
    "./kagglecatsanddogs_5340/PetImages/Cat",
    "./kagglecatsanddogs_5340/PetImages/Dog"
]

for folder in folders:
    for filename in os.listdir(folder):
        path = os.path.join(folder, filename)

        try:
            with Image.open(path) as img:
                img.verify()
        except Exception:
            print("Removing invalid image:", path)
            os.remove(path)

In [27]:
# train the model

epochs=10

train_model(
    model,
    trainloader,
    validationloader,
    epochs
)

Epoch [1/10] | Train Loss: 0.3875 | Train Accuracy: 82.64% | Validation Loss: 0.3903 | Validation Accuracy: 82.52%
Epoch [2/10] | Train Loss: 0.3380 | Train Accuracy: 85.43% | Validation Loss: 0.3742 | Validation Accuracy: 83.66%
Epoch [3/10] | Train Loss: 0.2892 | Train Accuracy: 87.81% | Validation Loss: 0.3958 | Validation Accuracy: 82.60%
Epoch [4/10] | Train Loss: 0.2427 | Train Accuracy: 89.96% | Validation Loss: 0.3641 | Validation Accuracy: 84.38%
Epoch [5/10] | Train Loss: 0.2030 | Train Accuracy: 91.51% | Validation Loss: 0.4244 | Validation Accuracy: 84.48%
Epoch [6/10] | Train Loss: 0.1682 | Train Accuracy: 93.18% | Validation Loss: 0.4130 | Validation Accuracy: 84.44%
Epoch [7/10] | Train Loss: 0.1334 | Train Accuracy: 94.87% | Validation Loss: 0.4470 | Validation Accuracy: 83.92%
Epoch [8/10] | Train Loss: 0.1071 | Train Accuracy: 95.88% | Validation Loss: 0.4646 | Validation Accuracy: 84.86%
Epoch [9/10] | Train Loss: 0.0952 | Train Accuracy: 96.26% | Validation Loss: 0.

In [35]:
def predict_image(image_path):

    # Open image and convert it to RGB
    image = Image.open(image_path).convert("RGB")

    # Apply same transformation used during training
    image = transform(image)

    # Add batch dimension
    image = image.unsqueeze(0)

    # Move image to CPU/GPU
    image = image.to(device)

    # loading the best model
    model.load_state_dict(torch.load("best_model.pt"))
    
    # Set model to evaluation mode
    model.eval()

    with torch.no_grad():

        output = model(image)

        # Convert logit to probability
        probability = torch.sigmoid(output).item()

    if probability >= 0.5:
        predicted_class = "Dog"
    else:
        predicted_class = "Cat"

    print("Predicted class:", predicted_class)
    print("Dog probability:", probability)

In [36]:
predict_image("./cat.jpg")

Predicted class: Cat
Dog probability: 0.04012908786535263


In [ ]:
print()